In [1]:
"""
Method 3: Magnitude Pruning — ResNet-50 / CIFAR-10
===================================================
Per-layer (local) magnitude pruning using L1 and L2 norms.
Each layer is pruned independently to the same target sparsity,
unlike global unstructured pruning where all layers are ranked together.

Compares three criteria: local-L1, local-L2, and global-L1.

Saves exactly 8 metrics per (sparsity × criterion) for comparison with baseline:
  accuracy, precision, recall, f1, num_parameters, model_size_mb,
  inference_time_ms, flops

Output: method3_magnitude_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method3_magnitude_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS = [0.30, 0.50, 0.70, 0.80, 0.90]
INFERENCE_RUNS  = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning criteria ──────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def apply_local_l1(model, sparsity):
    """Per-layer L1 unstructured magnitude pruning (local, not global)."""
    pruned = copy.deepcopy(model)
    for module, param in prunable_layers(pruned):
        prune.l1_unstructured(module, name=param, amount=sparsity)
        prune.remove(module, param)
    return pruned

def apply_local_l2(model, sparsity):
    """Per-layer L2 magnitude pruning — keeps largest L2-norm weights."""
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w         = module.weight.data
            flat      = w.pow(2).view(-1)
            threshold = torch.kthvalue(flat, max(1, int(flat.numel() * sparsity))).values
            mask      = (w.pow(2) > threshold).float()
            module.weight.data = w * mask
    return pruned

def apply_global_l1(model, sparsity):
    """Global L1 magnitude pruning — all layers ranked together."""
    pruned  = copy.deepcopy(model)
    layers  = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, param in layers:
        prune.remove(module, param)
    return pruned


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Method 3: Magnitude Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device : {DEVICE}")
    print(f"  Comparing: local-L1, local-L2, global-L1")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    CRITERIA = [
        ("local_l1",  apply_local_l1),
        ("local_l2",  apply_local_l2),
        ("global_l1", apply_global_l1),
    ]

    results = []

    for sparsity in SPARSITY_LEVELS:
        for criterion_name, prune_fn in CRITERIA:
            print(f"  Sparsity {int(sparsity*100)}% | Criterion: {criterion_name} ...")
            pruned = prune_fn(model, sparsity)

            metrics       = evaluate(pruned, loader)
            num_params    = count_parameters(pruned)
            model_size_mb = get_model_size_mb(pruned)
            inference_ms  = measure_inference_time_ms(pruned)
            flops         = compute_flops(pruned)

            row = {
                "sparsity" : sparsity,
                "criterion": criterion_name,
                # ── 8 standardized metrics ──
                "accuracy"         : round(metrics["accuracy"],  6),
                "precision"        : round(metrics["precision"], 6),
                "recall"           : round(metrics["recall"],    6),
                "f1"               : round(metrics["f1"],        6),
                "num_parameters"   : num_params,
                "model_size_mb"    : round(model_size_mb, 4),
                "inference_time_ms": round(inference_ms, 4),
                "flops"            : flops,
            }
            results.append(row)

            print(f"    Acc: {metrics['accuracy']:.4f} | Params: {num_params:,} | "
                  f"Size: {model_size_mb:.2f} MB | Inf: {inference_ms:.2f} ms | "
                  f"FLOPs: {flops/1e9:.3f} G")

    report = {
        "method"     : "magnitude_pruning",
        "description": "Per-layer magnitude pruning comparing local-L1, local-L2, and global-L1 criteria",
        "baseline"   : baseline,
        "results"    : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Method 3: Magnitude Pruning — ResNet-50 / CIFAR-10
  Device : cuda
  Comparing: local-L1, local-L2, global-L1

  Sparsity 30% | Criterion: local_l1 ...
    Acc: 0.9319 | Params: 23,520,842 | Size: 90.03 MB | Inf: 65.37 ms | FLOPs: 2.623 G
  Sparsity 30% | Criterion: local_l2 ...
    Acc: 0.9319 | Params: 23,520,842 | Size: 90.03 MB | Inf: 63.45 ms | FLOPs: 2.623 G
  Sparsity 30% | Criterion: global_l1 ...
    Acc: 0.9321 | Params: 23,520,842 | Size: 90.03 MB | Inf: 120.34 ms | FLOPs: 2.623 G
  Sparsity 50% | Criterion: local_l1 ...
    Acc: 0.9318 | Params: 23,520,842 | Size: 90.03 MB | Inf: 93.75 ms | FLOPs: 2.623 G
  Sparsity 50% | Criterion: local_l2 ...
    Acc: 0.9318 | Params: 23,520,842 | Size: 90.03 MB | Inf: 158.15 ms | FLOPs: 2.623 G
  Sparsity 50% | Criterion: global_l1 ...
    Acc: 0.9320 | Params: 23,520,842 | Size: 90.03 MB | Inf: 105.74 ms | FLOPs: 2.623 G
  Sparsity 70% | Criterion: local_l1 ...
    Acc: 0.9286 | Params: 23,520,842 | Size: 90.03 MB | Inf: 78.44 ms | 